In [ ]:
#!pip install selenium webdriver-manager python-dotenv

In [ ]:
import os
import time
import json as _json
from datetime import datetime

from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException


BASE_SEARCH_URL = (
    "https://www.ft.com/search?q=precious%20metals"
    "&contentType=article"
    "&dateFrom=2024-01-01"
    "&dateTo=2025-11-26"
)


def create_driver(headless: bool = False) -> webdriver.Chrome:
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


def accept_cookies_if_present(driver, timeout: int = 8):
    """Accept FT cookie banner if it appears."""
    try:
        wait = WebDriverWait(driver, timeout)
        cookie_button = wait.until(
            EC.element_to_be_clickable((
                By.CSS_SELECTOR,
                "button[title*='Accept'] , button[mode='primary'], button[aria-label*='Accept']"
            ))
        )
        cookie_button.click()
        time.sleep(1)
    except TimeoutException:
        pass


def login_ft(driver):
    load_dotenv()
    username = os.getenv("FT_USERNAME")
    password = os.getenv("FT_PASSWORD")

    driver.get("https://www.ft.com/login")
    wait = WebDriverWait(driver, 20)

    email_field = wait.until(EC.presence_of_element_located((By.XPATH, "/html/body/div[2]/div[2]/div/div[2]/div/form/span/span[2]/input")))
    email_field.clear() 
    email_field.send_keys(username)
    driver.find_element(By.CSS_SELECTOR, "#enter-email-next").click()

    password_field = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#enter-password")))
    password_field.clear(); password_field.send_keys(password)
    driver.find_element(By.CSS_SELECTOR, "#sign-in-button").click()
    time.sleep(25)



def open_search_page(driver, page: int = 1):
    driver.get(f"{BASE_SEARCH_URL}&page={page}")
    accept_cookies_if_present(driver)


def collect_result_links(driver):
    links = driver.find_elements(By.CSS_SELECTOR, "a.js-teaser-heading-link, a[data-trackable='heading']")
    return [lnk.get_attribute("href") for lnk in links if lnk.get_attribute("href")]


def scrape_article(driver, url: str):
    driver.get(url)
    wait = WebDriverWait(driver, 15)
    try:
        title_el = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
    except TimeoutException:
        return None
    title = title_el.text.strip()
    # publication date
    date_el = driver.find_elements(By.CSS_SELECTOR, "time[datetime]")
    date_iso = date_el[0].get_attribute("datetime") if date_el else ""
    paras = driver.find_elements(By.CSS_SELECTOR, ".o-topper__standfirst , #article-body p")
    content = "\n".join(p.text for p in paras if p.text)
    print(f"Scraped article: {title}, date: {date_iso}")
    return {
        "url": url,
        "title": title,
        "date": date_iso,
        "content": content
    }


def run_scrape(headless: bool = False, max_pages: int = 12, pause: float = 1.5):
    driver = create_driver(headless=headless)
    articles = []
    seen = set()
    try:
        login_ft(driver)
        for page in range(1, max_pages + 1):
            open_search_page(driver, page)
            time.sleep(2)
            urls = collect_result_links(driver)
            urls = [u for u in urls if u and u not in seen]
            if not urls:
                break
            for url in urls:
                seen.add(url)
                art = scrape_article(driver, url)
                if art:
                    articles.append(art)
                time.sleep(pause)
    finally:
        driver.quit()
    return articles


def save_jsonl(records, path: str):
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(_json.dumps(rec, ensure_ascii=False) + "\n")


def main():
    arts = run_scrape(headless=False, max_pages=12)
    save_jsonl(arts, "ft_precious_metals.jsonl")
    print(f"Saved {len(arts)} articles")


if __name__ == "__main__":
    main()

Scraped article: Germany shelves commemorative coins over surging silver price, date: 2025-10-23T04:00:28.217Z
Scraped article: Silver’s history points to why it is back in record territory, date: 2025-10-21T04:00:17.726Z
Scraped article: Silver price hits record amid scramble in London market, date: 2025-10-14T07:04:34.426Z
Scraped article: Silver, the precious metal ready to outshine gold, date: 2025-06-30T16:30:04.563Z
Scraped article: US adds silver and copper to ‘critical minerals’ list, date: 2025-11-06T18:13:13.554Z
Scraped article: Silverines cash in on a dysfunctional market, date: 2025-10-17T04:00:14.318Z
Scraped article: Silver’s growth brings back ethical questions that tarnish its shine, date: 2025-11-14T04:00:09.858Z
Scraped article: Singapore and Hong Kong go for gold in precious-metal trade, date: 2025-09-29T04:00:07.125Z
Scraped article: Silver and platinum prices soar as investors seek ‘gold alternatives’, date: 2025-06-12T04:00:52.825Z
Scraped article: Gold rally tem